# 02 · 임베딩·FIQA 입력 준비와 점검
완료 입력을 재사용하고 coverage·전처리·hash를 검증합니다. 새 입력이 필요할 때만 PREPARE_INPUTS=True로 실행합니다.
같은 whole-image resize producer를 사용하며 다른 평가 프로토콜을 실행하지 않습니다.
[실행 안내](README.md) · [지표·보존 안내](LFW_PAIR_VERIFICATION.md)

In [ ]:
CONFIG_PATH = "configs/experiments/lfw_pair_verification.yaml"
PREPARE_INPUTS = False  # True: CUDA로 resize/FR/FIQA 생성 또는 검증 후 재사용
INPUT_CONFIG_PATH = None  # None: pair YAML의 input_config_path 또는 기존 기본 입력 설정
DOWNLOAD_INPUT_INDEX = False  # 최초 입력 준비 시 필요한 고정 전체 이미지 목록
DEVICE = "cuda"
BATCH_SIZE = 32

In [ ]:
from pathlib import Path
import sys
import yaml
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "research").is_dir() and (p / "configs").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Python:", sys.executable)

from research.experiments.lfw_pair_verification import run_verification
if PREPARE_INPUTS:
    from research.experiments.lfw_resize_inputs import prepare_resize_inputs
    from research.datasets.lfw_blufr import fetch_blufr_config
    pair_config = yaml.safe_load((PROJECT_ROOT / CONFIG_PATH).read_text(encoding="utf-8"))
    resolved_input_config = INPUT_CONFIG_PATH or pair_config.get("input_config_path", "configs/experiments/lfw_blufr_calibration_resize.yaml")
    input_config = yaml.safe_load((PROJECT_ROOT / resolved_input_config).read_text(encoding="utf-8"))
    from research.experiments.model_bindings import model_bindings
    if model_bindings(pair_config) != model_bindings(input_config):
        raise ValueError("입력/평가 model UID 또는 차원 불일치")
    # 생성 위치와 다음 평가에서 읽을 위치가 다르면 계산 전에 중단합니다.
    for key in ("image_manifest", "fiqa_root", "fiqa_variant"):
        if pair_config[key] != input_config[key]:
            raise ValueError(f"입력/평가 설정 불일치: {key}")
    for model, source in pair_config["source_runs"].items():
        if source != input_config["source_runs"].get(model):
            raise ValueError(f"입력/평가 source 불일치: {model}")
    if DOWNLOAD_INPUT_INDEX:
        fetch_blufr_config(PROJECT_ROOT / input_config["protocol_file"],
            url=input_config["protocol_source_url"], expected_sha256=input_config["protocol_sha256"])
    input_result = prepare_resize_inputs(PROJECT_ROOT, config_path=resolved_input_config,
        models=tuple(pair_config["source_runs"]), execute=True, device=DEVICE,
        batch_size=BATCH_SIZE, resize_only=False)
    display(input_result)
pair_plan = run_verification(PROJECT_ROOT, config_path=CONFIG_PATH, execute=False)

In [ ]:
display(pair_plan["coverage"])
print("Ready:", pair_plan["ready"], "· 예정 jobs:", pair_plan["expected_jobs"])
if not pair_plan["ready"]:
    raise RuntimeError("입력 누락을 해결한 뒤 02를 재실행하세요. 누락 이미지를 제외하지 않습니다.")
print("다음: 03_calibration.ipynb")

입력 생성은 기존 모델 registry와 checkpoint를 필요로 하며 CUDA가 없으면 중단합니다.
입력 NPY·품질 CSV·재개 cache는 이후 평가의 재사용 입력으로 보존합니다.
03의 KEEP_RAW_RESULTS는 이 입력 파일을 삭제하지 않습니다. 입력 설정 파일명에 BLUFR가 남아 있지만 여기서는 전체 이미지 대응 및 resize 입력 생성에만 사용합니다.